# F2 · 03 · Validación técnica

**Qué hace este notebook:** comprueba de forma sistemática que el dato y el código se comportan como esperamos, en casos normales, casos límite y excepciones, y deja evidencia de ello.
**Qué no hace:** análisis entre grupos (Fase 3).

La lógica está en `src/arena/validation.py` y las pruebas automáticas en `tests/`.

## 0. Entorno
Misma verificación que en los otros notebooks: versiones fijadas frente a instaladas, rama y commit.

In [1]:
from arena.env import environment_info

info = environment_info()
{k: v for k, v in info.items() if k != "dependencies"}

{'python': '3.12.13',
 'system': 'Darwin 25.6.0',
 'interpreter': '/Users/jose/Developer/magister/programacion_ds/lmarena-sumativa1/.venv/bin/python',
 'virtual_env': True,
 'seed': 42,
 'git': {'branch': 'main', 'commit': '0ccff6a', 'clean': 'False'}}

In [2]:
import pandas as pd

pd.DataFrame(info["dependencies"]).T

,pinned,installed,match
numpy,2.5.4,2.5.4,True
pandas,3.1.0,3.1.0,True
pyarrow,26.0.0,26.0.0,True
matplotlib,3.11.2,3.11.2,True
huggingface_hub,2.4.0,2.4.0,True
pytest,9.2.0,9.2.0,True
nbconvert,7.17.2,7.17.2,True
nbformat,5.11.1,5.11.1,True
ipykernel,7.4.0,7.4.0,True
jupyter_client,8.10.0,8.10.0,True


*Interpretación:* todas las dependencias instaladas coinciden con las fijadas en `requirements.txt` (`match: True`). Queda registrado el commit desde el que se ejecutó.

## 1. Qué se valida y por qué
**Qué hacemos:** definir los chequeos que debe cumplir el dato antes de usarlo en la Fase 3.
**Por qué:** un error silencioso (un ganador inválido, un id repetido, tokens negativos) contaminaría cualquier análisis posterior sin dar señales.

| Chequeo | Tipo | Motivo |
|---|---|---|
| Columnas requeridas presentes | crítico | Sin ellas no hay análisis posible |
| Sin NA en columnas clave | crítico | Evita filas incompletas |
| `id` único | crítico | Un voto contado dos veces sesga las proporciones |
| `winner` en {`model_a`, `model_b`, `tie`, `both_bad`} | crítico | Son los únicos resultados posibles |
| `evaluation_order` entero entre 1 y 28 | crítico | Rango observado en el dato |
| `timestamp` entre el 17-abr y el 24-jul de 2025 | crítico | Ventana temporal del dataset |
| Tokens y conteos ≥ 0 | crítico | Un conteo negativo no tiene sentido |
| Filas conservadas respecto del dato original | crítico | Cualquier fila eliminada debe estar documentada |
| `model_a` distinto de `model_b` | advertencia | Caso raro, no invalida el resto del dato |

Los chequeos **críticos** lanzan `ValueError` con `assert_ok()`; las **advertencias** se informan pero no detienen el flujo.

## 2. Validación del dato original (aplanado)
**Qué hacemos:** cargar el dato, aplanar `conv_metadata` y `category_tag` solo para esta revisión, y correr `validate_dataset`.
**Qué esperamos:** ningún fallo crítico y, como máximo, la advertencia de las 10 filas con el mismo modelo en ambos lados que ya vimos en F2_01.

In [3]:
from arena.profile import count_rows, flat_view, load_raw
from arena.validation import validate_dataset

df = flat_view(load_raw())
report = validate_dataset(df, expected_rows=count_rows()["total"])
report.to_frame()

,status,check,detail
0,OK,required columns,all present
1,OK,no NA in key columns,none
2,OK,unique id,none duplicated
3,OK,winner in allowed set,all valid
4,OK,"evaluation_order integer in [1, 28]",0 out of range
5,OK,timestamp within 2025-04-17..2025-07-24,0 outside
6,OK,sum_assistant_a_tokens >= 0,0 negative
7,OK,sum_assistant_b_tokens >= 0,0 negative
8,OK,sum_user_tokens >= 0,0 negative
9,OK,context_a_tokens >= 0,0 negative


In [4]:
report.assert_ok()
{"critical_failures": len(report.failures), "warnings": len(report.warnings)}

{'critical_failures': 0, 'warnings': 1}

*Interpretación:* los chequeos críticos pasan y `assert_ok()` no lanza excepción: el dato original es coherente. Queda una única advertencia, las 10 filas con `model_a == model_b`. Se conserva el número de filas (135.634). La decisión sobre esas 10 filas se toma en la limpieza.

## 3. Casos límite y excepciones
**Qué hacemos:** provocar errores a propósito en una copia pequeña del dato y comprobar que la validación los detecta.
**Por qué:** una validación que nunca falla no demuestra nada; hay que ver que sí reacciona.
**Qué esperamos:** cada error provocado aparece como `FAIL` y `assert_ok()` lanza `ValueError` con un mensaje claro.

In [5]:
broken = df.head(1000).copy()
broken.loc[0, "winner"] = "draw"          # invalid result
broken.loc[1, "sum_assistant_a_tokens"] = -5  # impossible count
broken.loc[2, "id"] = broken.loc[3, "id"]    # repeated id
broken_report = validate_dataset(broken)
broken_report.to_frame().query("status != 'OK'")

,status,check,detail
2,FAIL,unique id,1 duplicated
3,FAIL,winner in allowed set,invalid: ['draw']
6,FAIL,sum_assistant_a_tokens >= 0,1 negative


In [6]:
try:
    broken_report.assert_ok()
except ValueError as error:
    message = str(error)
message

"3 critical validation check(s) failed: unique id: 1 duplicated; winner in allowed set: invalid: ['draw']; sum_assistant_a_tokens >= 0: 1 negative"

*Interpretación:* los tres errores provocados se detectan como `FAIL`, cada uno con su nombre y detalle, y `assert_ok()` lanza un `ValueError` que los lista todos juntos. La advertencia del mismo modelo en ambos lados no aparece aquí porque esas 10 filas no están en las primeras 1.000.

In [7]:
missing = validate_dataset(df.drop(columns=["winner"]).head(10))
missing.to_frame().head(3)

,status,check,detail
0,FAIL,required columns,missing: ['winner']
1,OK,no NA in key columns,none
2,OK,unique id,none duplicated


*Interpretación:* si falta una columna, el primer chequeo lo informa por nombre y los chequeos que la necesitan se omiten en lugar de fallar con un error confuso.

## 4. Pruebas automáticas
**Qué hacemos:** mostrar la salida guardada de `pytest`, que se genera con `make test` y se versiona en `docs/evidence/pytest.txt`.
**Qué cubren:** casos normales (datos válidos), límites (`evaluation_order` en 1 y 28, 0 tokens, sesión de un voto) y excepciones (columna faltante, ganador inválido, tokens negativos, id repetido, entrada que no es DataFrame, archivo de huellas inexistente).

In [8]:
from arena.env import read_evidence

print(read_evidence("pytest.txt"))

.......................................................                  [100%]
55 passed in 0.60s



*Interpretación:* todas las pruebas pasan, no usan la red y se ejecutan en menos de un segundo. La salida queda guardada como evidencia verificable.

## 5. Validación del dataset procesado
**Qué hacemos:** cargar el archivo limpio generado en `F2_02` (`data/processed/arena_clean.parquet`), correr `validate_dataset` verificando conservación de filas (`expected_rows=135_634`) y tipos objetivo (`expected_dtypes`), y comprobar la consistencia interna de las 7 variables derivadas.
**Qué esperamos:** 0 fallos críticos, conservación exacta de las 135.634 filas × 54 columnas y consistencia matemática entre los conteos base y las variables derivadas.

In [9]:
from arena.preprocessing import OUTPUT_NAME, PROCESSED_DIR

clean_path = PROCESSED_DIR / OUTPUT_NAME
df_clean = pd.read_parquet(clean_path)
target_dtypes = {
    "winner": "category",
    "language": "category",
    "evaluation_order": "int16",
    "is_code": "bool",
}
clean_report = validate_dataset(
    df_clean,
    expected_rows=count_rows()["total"],
    expected_dtypes=target_dtypes,
)
clean_report.assert_ok()
clean_report.to_frame()

,status,check,detail
0,OK,required columns,all present
1,OK,no NA in key columns,none
2,OK,unique id,none duplicated
3,OK,winner in allowed set,all valid
4,OK,"evaluation_order integer in [1, 28]",0 out of range
5,OK,timestamp within 2025-04-17..2025-07-24,0 outside
6,OK,sum_assistant_a_tokens >= 0,0 negative
7,OK,sum_assistant_b_tokens >= 0,0 negative
8,OK,sum_user_tokens >= 0,0 negative
9,OK,context_a_tokens >= 0,0 negative


In [10]:
import numpy as np

derived_checks = {
    "shape_135634x54": df_clean.shape == (135_634, 54),
    "no_na_derived": df_clean[[
        "format_a", "format_b", "tokens_log_a", "tokens_log_b",
        "longer_response", "more_formatted", "winner_binary"
    ]].isna().sum().sum() == 0,
    "tokens_log_a_matches_log1p": np.allclose(
        df_clean["tokens_log_a"],
        np.log1p(df_clean["sum_assistant_a_tokens"]).astype("float32"),
    ),
    "longer_response_consistent": (
        (df_clean["longer_response"] == "a")
        == (df_clean["sum_assistant_a_tokens"] > df_clean["sum_assistant_b_tokens"])
    ).all(),
    "more_formatted_consistent": (
        (df_clean["more_formatted"] == "a")
        == (df_clean["format_a"] > df_clean["format_b"])
    ).all(),
}
assert all(derived_checks.values()), f"Fallo en derivadas: {derived_checks}"
derived_checks

{'shape_135634x54': True,
 'no_na_derived': np.True_,
 'tokens_log_a_matches_log1p': True,
 'longer_response_consistent': np.True_,
 'more_formatted_consistent': np.True_}

*Interpretación:* el dataset procesado (`data/processed/arena_clean.parquet`, 135.634 filas × 54 columnas) supera todos los chequeos críticos, conserva los tipos objetivo (`category`, `int16`, `bool`, `datetime64[ns]`) y mantiene consistencia exacta en las 7 variables derivadas.

## 6. Resumen
- El dato original supera todos los chequeos críticos y tiene una advertencia conocida (10 filas con `model_a == model_b`).
- La validación detecta los errores provocados y los informa con mensajes claros.
- Las pruebas automáticas con `pytest` pasan en su totalidad y su salida queda guardada en `docs/evidence/pytest.txt`.
- El dataset procesado (`data/processed/arena_clean.parquet`, 135.634 filas × 54 columnas) supera todos los chequeos de integridad, tipos y consistencia de variables derivadas, quedando listo para la Fase 3.